# Text Preprocessing for Clinical Notes

Time estimate: **30** minutes

## Objectives

After completing this lab, you will be able to:

 - Clean and normalize raw clinical notes to prepare them for analysis
 - Tokenize clinical text and apply text preprocessing techniques
 - Extract named entities from clinical documents using NER models

## What you will do in this lab

Clinical notes contain valuable information but require significant preprocessing before they can be analyzed effectively. In this lab, you will work with raw clinical notes to transform them into structured, analyzable data.

Throughout this lab, you will clean messy text data, apply various normalization techniques, and extract meaningful medical entities from unstructured clinical narratives. These skills form the foundation for any clinical natural language processing project.

You will:

- Load and examine raw clinical note data
- Clean text by removing special characters and normalizing whitespace
- Tokenize clinical notes into words and sentences
- Apply lemmatization and stopword removal techniques
- Extract medical named entities such as diseases, symptoms, and medications

## Overview

Clinical notes are narrative text documents written by healthcare providers that contain critical information about patient encounters, diagnoses, treatments, and outcomes. However, these notes are often filled with abbreviations, inconsistent formatting, and specialized medical terminology that makes them challenging to process computationally.

Text preprocessing is the essential first step in any clinical NLP pipeline. It involves transforming raw, unstructured text into a clean, standardized format that can be analyzed by machine learning models. This process includes removing noise, normalizing text variations, and breaking down complex documents into manageable units.

Named Entity Recognition (NER) is a crucial technique in clinical text processing that identifies and classifies important medical concepts within the text. By automatically extracting entities like diseases, medications, and symptoms, NER enables researchers and clinicians to quickly identify key information from large volumes of clinical documentation.

In this lab, you will apply industry-standard preprocessing techniques using Python libraries specifically designed for natural language processing and clinical text analysis. These techniques are used in real-world healthcare applications ranging from clinical decision support systems to medical research and population health management.

## About the dataset

For this lab, you will work with synthetic clinical notes that simulate real electronic health record documentation.

### Dataset overview

The dataset contains anonymized clinical notes representing various types of medical encounters. These notes have been created to reflect the structure and content of actual clinical documentation while protecting patient privacy. Each note includes typical elements found in medical records such as patient histories, physical examination findings, diagnoses, and treatment plans.

### Column descriptions

1. **note_id** - Unique identifier for each clinical note
2. **patient_id** - Anonymous identifier linking the note to a specific patient
3. **note_type** - Category of clinical documentation (e.g., Progress Note, Discharge Summary, History and Physical)
4. **note_text** - The raw, unprocessed clinical narrative text
5. **note_date** - Date when the clinical note was created
6. **provider_specialty** - Medical specialty of the healthcare provider who authored the note

## Setup

### Installing required libraries

The following libraries are required to run this lab.

In [1]:
# Install the libraries required for this lab
!pip install pandas
!pip install nltk
!pip install regex

In [2]:
# Download required NLTK data
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [1]:
# Install spaCy (compatible version)
!pip install spacy==3.7.2

In [2]:


# Install scispacy
!pip install scispacy

# Install the scientific model (latest version)
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

# Restart runtime (important!)
# Go to Runtime > Restart runtime in Colab menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 34.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 42.4 MB/s eta 0:00:00
  Created wheel for en_core_sci_sm: filename=en_core_sci_sm-0.5.4-py3-none-any.whl size=14778488 sha256=41b70380432e3840f26ff825fc01558ac65925a436eb808dfa93c08757c0914b
  Stored in directory: /root/.cache/pip/wheels/49/7f/0f/ec0fc3a935bfe55e6ef2ca04b7a31e33cbd533a6d7cbd9e11e
Successfully built en_core_sci_sm
  Attempting uninstall: spacy
    Found existing installation: spacy 3.7.2
    Uninstalling spacy-3.7.2:
      Successfully uninstalled spacy-3.7.2


In [3]:
# Optional: suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Importing required libraries

In [4]:
# Import pandas for data manipulation and analysis
import pandas as pd

# Import regex for advanced pattern matching
import re

# Import NLTK components for text processing
from nltk.tokenize import word_tokenize, sent_tokenize  # for breaking text into tokens
from nltk.corpus import stopwords  # for removing common words
from nltk.stem import WordNetLemmatizer  # for word normalization

# Import spaCy for advanced NLP and NER
import spacy

# Load the medical-specific NER model
nlp = spacy.load('en_core_sci_sm')

print("All libraries imported successfully!")
print("Ready to begin clinical text preprocessing analysis.")

All libraries imported successfully!
Ready to begin clinical text preprocessing analysis.


## Step 1: Load and examine clinical notes

Before you can process clinical notes, you need to understand their structure and content. In this step, you will create a sample dataset and examine the raw clinical text to identify preprocessing challenges such as inconsistent formatting, special characters, and medical abbreviations.

In [5]:
# Create sample clinical notes dataset
clinical_notes = {
    'note_id': [1, 2, 3],
    'patient_id': ['P001', 'P002', 'P003'],
    'note_type': ['Progress Note', 'Discharge Summary', 'History and Physical'],
    'note_text': [
        """Patient presents with c/o severe headache and nausea x 3 days.
        PMH: HTN, DM Type II.
        Meds: Metformin 500mg BID, Lisinopril 10mg QD.
        PE: BP 145/92, HR 88, Temp 98.6F. Patient appears uncomfortable.
        Assessment: Migraine headache, uncontrolled hypertension.
        Plan: Start Sumatriptan 50mg PRN, increase Lisinopril to 20mg QD.""",

        """DISCHARGE SUMMARY: 65 y/o M admitted for COPD exacerbation.
        Patient with h/o smoking (40 pack-years), presented with SOB and productive cough.
        Hospital course: Treated with nebulizers, prednisone 40mg daily, and antibiotics (Azithromycin).
        Condition improved significantly.
        Discharge meds: Albuterol inhaler, Advair 250/50 BID, Prednisone taper.
        F/U with pulmonology in 2 weeks.""",

        """CC: Chest pain. HPI: 58 y/o F w/ sudden onset chest pain radiating to L arm, assoc. w/ diaphoresis.
        PMH: Hyperlipidemia, Family h/o CAD.
        Exam: Patient diaphoretic, anxious. CV: Regular rate and rhythm, no murmurs.
        EKG: ST elevation in leads II, III, aVF.
        Impression: Acute inferior STEMI.
        Plan: Activate cath lab, start heparin drip, aspirin 325mg, plavix load 600mg."""
    ],
    'note_date': ['2024-01-15', '2024-01-20', '2024-01-22'],
    'provider_specialty': ['Internal Medicine', 'Pulmonology', 'Cardiology']
}

# Create DataFrame from the clinical notes data
df = pd.DataFrame(clinical_notes)

# Display the dataset structure
print("Clinical Notes Dataset:")
print(f"Total number of notes: {len(df)}")
print(f"\nDataset columns: {list(df.columns)}")
print("\nFirst clinical note:")
print(df.iloc[0]['note_text'])

Clinical Notes Dataset:
Total number of notes: 3

Dataset columns: ['note_id', 'patient_id', 'note_type', 'note_text', 'note_date', 'provider_specialty']

First clinical note:
Patient presents with c/o severe headache and nausea x 3 days.
        PMH: HTN, DM Type II.
        Meds: Metformin 500mg BID, Lisinopril 10mg QD.
        PE: BP 145/92, HR 88, Temp 98.6F. Patient appears uncomfortable.
        Assessment: Migraine headache, uncontrolled hypertension.
        Plan: Start Sumatriptan 50mg PRN, increase Lisinopril to 20mg QD.


In [ ]:
#display top rows from dataframe
df.head()

## Step 2: Text cleaning and normalization

Raw clinical notes contain various formatting inconsistencies that can interfere with analysis. This step removes unwanted characters, normalizes whitespace, and converts text to a consistent format. Text cleaning is critical because it ensures that downstream processing steps can focus on meaningful content rather than formatting artifacts.

In [7]:
def clean_clinical_text(text):
    """
    Clean and normalize clinical note text.

    Parameters:
    text (str): Raw clinical note text

    Returns:
    str: Cleaned and normalized text
    """
    # Convert to lowercase for consistency
    text = text.lower()

    # Remove multiple spaces and normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove leading and trailing whitespace
    text = text.strip()

    # Replace newline characters with spaces
    text = text.replace('\n', ' ')

    # Remove special characters but preserve medical notation (keep periods, commas, slashes, hyphens)
    text = re.sub(r'[^a-z0-9\s.,/\-]', '', text)

    return text

# Apply cleaning function to all clinical notes
df['cleaned_text'] = df['note_text'].apply(clean_clinical_text)

# Display before and after comparison
print("Original text:")
print(df.iloc[0]['note_text'][:200])
print("\nCleaned text:")
print(df.iloc[0]['cleaned_text'][:200])
print("\nText cleaning completed successfully!")

Original text:
Patient presents with c/o severe headache and nausea x 3 days.
        PMH: HTN, DM Type II.
        Meds: Metformin 500mg BID, Lisinopril 10mg QD.
        PE: BP 145/92, HR 88, Temp 98.6F. Patient ap

Cleaned text:
patient presents with c/o severe headache and nausea x 3 days. pmh htn, dm type ii. meds metformin 500mg bid, lisinopril 10mg qd. pe bp 145/92, hr 88, temp 98.6f. patient appears uncomfortable. assess

Text cleaning completed successfully!


## Step 3: Text tokenization

Tokenization is the process of breaking text into individual units called tokens. For clinical notes, you can perform both sentence tokenization (splitting text into sentences) and word tokenization (splitting text into individual words). This step is fundamental because most NLP algorithms require text to be in tokenized form for processing.

Sentence tokenization helps preserve context and meaning by keeping related information together, while word tokenization enables you to analyze individual terms and their frequencies.

In [8]:
def tokenize_text(text):
    """
    Tokenize clinical text into sentences and words.

    Parameters:
    text (str): Cleaned clinical note text

    Returns:
    dict: Dictionary containing sentence and word tokens
    """
    # Split text into individual sentences
    sentences = sent_tokenize(text)

    # Split text into individual words
    words = word_tokenize(text)

    return {
        'sentences': sentences,
        'words': words
    }

# Apply tokenization to the first note as example
sample_tokens = tokenize_text(df.iloc[0]['cleaned_text'])

# Display tokenization results
print("Sentence Tokenization:")
print(f"Number of sentences: {len(sample_tokens['sentences'])}")
print("\nFirst sentence:")
print(sample_tokens['sentences'][0])

print("\n" + "="*50)
print("\nWord Tokenization:")
print(f"Total number of words: {len(sample_tokens['words'])}")
print("\nFirst 15 words:")
print(sample_tokens['words'][:15])

Sentence Tokenization:
Number of sentences: 7

First sentence:
patient presents with c/o severe headache and nausea x 3 days.


Word Tokenization:
Total number of words: 61

First 15 words:
['patient', 'presents', 'with', 'c/o', 'severe', 'headache', 'and', 'nausea', 'x', '3', 'days', '.', 'pmh', 'htn', ',']


## Step 4: Stopword removal and lemmatization

Not all words in clinical text carry equal importance. Stopwords are common words such as "the", "is", and "and" that occur frequently but provide little semantic value. Removing them reduces noise and focuses analysis on meaningful medical terms.

Lemmatization is the process of reducing words to their base or dictionary form. For example, "running", "runs", and "ran" all become "run". This normalization technique helps ensure that different forms of the same word are treated as a single concept, improving the quality of downstream analysis.

The combination of these techniques significantly reduces the dimensionality of the text while preserving its core meaning.

In [9]:
def preprocess_tokens(tokens):
    """
    Remove stopwords and apply lemmatization to tokens.

    Parameters:
    tokens (list): List of word tokens

    Returns:
    list: Preprocessed tokens
    """
    # Initialize lemmatizer to convert words to base form
    lemmatizer = WordNetLemmatizer()

    # Get English stopwords set
    stop_words = set(stopwords.words('english'))

    # Process each token
    processed_tokens = []
    for token in tokens:
        # Skip if token is a stopword
        if token.lower() in stop_words:
            continue

        # Skip if token is not alphabetic (removes punctuation and numbers separately)
        if not token.isalpha():
            continue

        # Apply lemmatization to reduce word to base form
        lemmatized_token = lemmatizer.lemmatize(token.lower())

        # Keep tokens that are longer than 2 characters
        if len(lemmatized_token) > 2:
            processed_tokens.append(lemmatized_token)

    return processed_tokens

# Apply preprocessing to sample tokens
processed_words = preprocess_tokens(sample_tokens['words'])

# Display preprocessing results
print("Original token count:", len(sample_tokens['words']))
print("After stopword removal and lemmatization:", len(processed_words))
print(f"Reduction: {len(sample_tokens['words']) - len(processed_words)} tokens removed")
print("\nProcessed tokens (first 20):")
print(processed_words[:20])

# Apply to all notes in dataset
df['tokens'] = df['cleaned_text'].apply(lambda x: preprocess_tokens(word_tokenize(x)))
print("\nPreprocessing applied to all clinical notes in dataset.")

Original token count: 61
After stopword removal and lemmatization: 28
Reduction: 33 tokens removed

Processed tokens (first 20):
['patient', 'present', 'severe', 'headache', 'nausea', 'day', 'pmh', 'htn', 'type', 'med', 'metformin', 'bid', 'lisinopril', 'temp', 'patient', 'appears', 'uncomfortable', 'assessment', 'migraine', 'headache']

Preprocessing applied to all clinical notes in dataset.


## Step 5: Named entity recognition (NER)

Named entity recognition is the process of identifying and classifying key medical concepts within clinical text. Using a specialized medical NER model, you can automatically extract important entities such as diseases, symptoms, medications, procedures, and anatomical structures.

The medical NER model has been trained on scientific and clinical literature, making it particularly effective at recognizing medical terminology. This automated extraction capability is essential for clinical decision support, medical coding, and large-scale clinical research applications.

In [10]:
def extract_medical_entities(text):
    """
    Extract named entities from clinical text using medical NER model.

    Parameters:
    text (str): Clinical note text

    Returns:
    list: List of dictionaries containing entity text and label
    """
    # Process the text with the medical NLP model
    doc = nlp(text)

    # Extract entities and their types
    entities = []
    for entity in doc.ents:
        entities.append({
            'text': entity.text,
            'label': entity.label_,
            'start': entity.start_char,
            'end': entity.end_char
        })

    return entities

# Extract entities from all clinical notes
df['entities'] = df['note_text'].apply(extract_medical_entities)

# Display entities from the first note
print("Named Entities Extracted from First Clinical Note:")
print("="*60)
for entity in df.iloc[0]['entities']:
    print(f"Entity: {entity['text']:<30} | Type: {entity['label']}")

# Display summary statistics
print("\n" + "="*60)
print("Entity Extraction Summary:")
for idx, row in df.iterrows():
    print(f"Note {row['note_id']}: {len(row['entities'])} entities found")

Named Entities Extracted from First Clinical Note:
Entity: Patient                        | Type: ENTITY
Entity: c/o severe headache            | Type: ENTITY
Entity: nausea x 3                     | Type: ENTITY
Entity: days                           | Type: ENTITY
Entity: PMH                            | Type: ENTITY
Entity: HTN                            | Type: ENTITY
Entity: DM Type II                     | Type: ENTITY
Entity: Meds                           | Type: ENTITY
Entity: Metformin                      | Type: ENTITY
Entity: BID                            | Type: ENTITY
Entity: QD                             | Type: ENTITY
Entity: PE                             | Type: ENTITY
Entity: BP                             | Type: ENTITY
Entity: HR                             | Type: ENTITY
Entity: Temp 98.6F                     | Type: ENTITY
Entity: Patient                        | Type: ENTITY
Entity: Assessment                     | Type: ENTITY
Entity: Migraine headache      

## Step 6: Create preprocessed dataset

Now that you have completed all preprocessing steps, you will consolidate the results into a final dataset. This dataset contains both the original and processed versions of each clinical note, allowing for easy comparison and understanding of the transformations applied. The preprocessed data is now ready for downstream analysis tasks such as classification, clustering, or information retrieval.

In [11]:
# Create summary of preprocessing results
summary_df = pd.DataFrame({
    'note_id': df['note_id'],
    'note_type': df['note_type'],
    'original_length': df['note_text'].apply(len),
    'cleaned_length': df['cleaned_text'].apply(len),
    'token_count': df['tokens'].apply(len),
    'entity_count': df['entities'].apply(len)
})

# Display summary
print("Preprocessing Summary:")
print(summary_df.to_string(index=False))

# Calculate overall statistics
print("\n" + "="*60)
print("Overall Statistics:")
print(f"Total notes processed: {len(df)}")
print(f"Average tokens per note: {summary_df['token_count'].mean():.1f}")
print(f"Average entities per note: {summary_df['entity_count'].mean():.1f}")
print(f"Total entities extracted: {summary_df['entity_count'].sum()}")

print("\nPreprocessed dataset is ready for analysis!")

Preprocessing Summary:
 note_id            note_type  original_length  cleaned_length  token_count  entity_count
       1        Progress Note              360             315           28            24
       2    Discharge Summary              418             371           32            27
       3 History and Physical              407             359           42            31

Overall Statistics:
Total notes processed: 3
Average tokens per note: 34.0
Average entities per note: 27.3
Total entities extracted: 82

Preprocessed dataset is ready for analysis!


# Exercises

Now that you have learned the fundamental techniques for preprocessing clinical notes, test your understanding with the following exercises.

## Exercise 1: Create and preprocess a new clinical note

Create a new clinical note about a patient with diabetes and apply the complete preprocessing pipeline (cleaning, tokenization, and stopword removal).

In [ ]:
new_note = """Patient is a 45 y/o M with Type 2 Diabetes Mellitus presenting for routine follow-up.
HbA1c today is 7.8%, up from 7.2% three months ago. Patient reports poor dietary compliance.
Current medications: Metformin 1000mg BID, Glipizide 5mg daily.
Plan: Refer to diabetes educator, increase Metformin to 1000mg TID, recheck HbA1c in 3 months."""

# your code goes here

<details>
    <summary>Click here for a hint</summary>
    
Follow the pattern from Steps 2, 3, and 4. First create a note string, then apply the `clean_clinical_text()` function, followed by `word_tokenize()`, and finally `preprocess_tokens()`.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Create a new clinical note about diabetes
new_note = """Patient is a 45 y/o M with Type 2 Diabetes Mellitus presenting for routine follow-up.
HbA1c today is 7.8%, up from 7.2% three months ago. Patient reports poor dietary compliance.
Current medications: Metformin 1000mg BID, Glipizide 5mg daily.
Plan: Refer to diabetes educator, increase Metformin to 1000mg TID, recheck HbA1c in 3 months."""

# Apply cleaning
cleaned_note = clean_clinical_text(new_note)
print("Cleaned note:")
print(cleaned_note)

# Apply tokenization
note_tokens = word_tokenize(cleaned_note)
print(f"\nTotal tokens: {len(note_tokens)}")

# Apply stopword removal and lemmatization
final_tokens = preprocess_tokens(note_tokens)
print(f"Preprocessed tokens: {len(final_tokens)}")
print("\nFinal tokens:")
print(final_tokens)
```

</details>

## Exercise 2: Extract and analyze entities by type

Using the second clinical note from the dataset (index 1), extract all named entities and create a dictionary that groups them by entity type.

In [ ]:
# your code goes here

<details>
    <summary>Click here for a hint</summary>
    
Access the second note using `df.iloc[1]['entities']`. Loop through the entities and group them by their 'label' field into a dictionary.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Get entities from second note
second_note_entities = df.iloc[1]['entities']

# Create dictionary to group entities by type
entities_by_type = {}

# Loop through entities and group by label
for entity in second_note_entities:
    label = entity['label']
    text = entity['text']
    
    # Add entity type to dictionary if not present
    if label not in entities_by_type:
        entities_by_type[label] = []
    
    # Append entity text to appropriate list
    entities_by_type[label].append(text)

# Display grouped entities
print("Entities grouped by type:")
print("="*60)
for entity_type, entity_list in entities_by_type.items():
    print(f"\n{entity_type}:")
    for entity in entity_list:
        print(f"  - {entity}")
```

</details>

## Exercise 3: Compare token frequencies across notes

Calculate and compare the frequency of the top 10 most common tokens across all three clinical notes in the dataset.

In [ ]:
# your code goes here


In [13]:
from collections import Counter

# Combine all tokens from all notes
all_tokens = []
for token_list in df['tokens']:
    all_tokens.extend(token_list)

# Count token frequencies
token_frequencies = Counter(all_tokens)

# Get top 10 most common tokens
top_10_tokens = token_frequencies.most_common(10)

# Display results
print("Top 10 Most Frequent Tokens Across All Notes:")
print("="*60)
print(f"{'Token':<20} {'Frequency':>10}")
print("="*60)
for token, frequency in top_10_tokens:
    print(f"{token:<20} {frequency:>10}")

print(f"\nTotal unique tokens: {len(token_frequencies)}")
print(f"Total tokens processed: {len(all_tokens)}")

Top 10 Most Frequent Tokens Across All Notes:
Token                 Frequency
patient                       4
headache                      2
pmh                           2
med                           2
bid                           2
lisinopril                    2
plan                          2
start                         2
discharge                     2
prednisone                    2

Total unique tokens: 88
Total tokens processed: 102


<details>
    <summary>Click here for a hint</summary>
    
Use Python's `Counter` class from the `collections` module. Combine all tokens from the 'tokens' column, then use `Counter.most_common(10)` to find the top 10.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Import Counter for frequency counting
from collections import Counter

# Combine all tokens from all notes
all_tokens = []
for token_list in df['tokens']:
    all_tokens.extend(token_list)

# Count token frequencies
token_frequencies = Counter(all_tokens)

# Get top 10 most common tokens
top_10_tokens = token_frequencies.most_common(10)

# Display results
print("Top 10 Most Frequent Tokens Across All Notes:")
print("="*60)
print(f"{'Token':<20} {'Frequency':>10}")
print("="*60)
for token, frequency in top_10_tokens:
    print(f"{token:<20} {frequency:>10}")

print(f"\nTotal unique tokens: {len(token_frequencies)}")
print(f"Total tokens processed: {len(all_tokens)}")
```

</details>

## Exercise 4: Build a custom preprocessing function

Create a single comprehensive function that takes raw clinical text as input and returns a dictionary containing cleaned text, tokens, and extracted entities.

## Exercise 4: Build a custom preprocessing function

Create a single comprehensive function that takes raw clinical text as input and returns a dictionary containing cleaned text, tokens, and extracted entities.

In [14]:
def full_preprocessing_pipeline(raw_text):
    """
    Complete preprocessing pipeline for clinical notes.

    Parameters:
    raw_text (str): Raw clinical note text

    Returns:
    dict: Dictionary containing all preprocessing results
    """
    # Step 1: Clean the text
    cleaned = clean_clinical_text(raw_text)

    # Step 2: Tokenize
    tokens = word_tokenize(cleaned)

    # Step 3: Remove stopwords and lemmatize
    processed_tokens = preprocess_tokens(tokens)

    # Step 4: Extract named entities from original text
    entities = extract_medical_entities(raw_text)

    # Return comprehensive results
    return {
        'cleaned_text': cleaned,
        'tokens': processed_tokens,
        'token_count': len(processed_tokens),
        'entities': entities,
        'entity_count': len(entities)
    }

In [ ]:
# Test the function with a sample note
test_note = """Patient presents with acute dyspnea and chest pain.
History of CAD and recent MI. Started on aspirin and statin therapy."""

# your code goes here

In [16]:
test_note = """Patient presents with acute dyspnea and chest pain.
History of CAD and recent MI. Started on aspirin and statin therapy."""

# Apply the full pipeline
results = full_preprocessing_pipeline(test_note)

# Display results
print("Preprocessing Pipeline Results:")
print("="*60)
print(f"Cleaned text: {results['cleaned_text']}")
print(f"\nToken count: {results['token_count']}")
print(f"Tokens: {results['tokens']}")
print(f"\nEntity count: {results['entity_count']}")
print("\nEntities extracted:")
for entity in results['entities']:
    print(f"  - {entity['text']} ({entity['label']})")

Preprocessing Pipeline Results:
Cleaned text: patient presents with acute dyspnea and chest pain. history of cad and recent mi. started on aspirin and statin therapy.

Token count: 13
Tokens: ['patient', 'present', 'acute', 'dyspnea', 'chest', 'pain', 'history', 'cad', 'recent', 'started', 'aspirin', 'statin', 'therapy']

Entity count: 7

Entities extracted:
  - Patient (ENTITY)
  - acute dyspnea (ENTITY)
  - chest pain (ENTITY)
  - CAD (ENTITY)
  - MI (ENTITY)
  - aspirin (ENTITY)
  - statin therapy (ENTITY)


<details>
    <summary>Click here for a hint</summary>
    
Combine the functions from Steps 2, 4, and 5. Your function should call `clean_clinical_text()`, `word_tokenize()`, `preprocess_tokens()`, and `extract_medical_entities()` in sequence.

</details>

<details>
    <summary>Click here for solution</summary>

```python
def full_preprocessing_pipeline(raw_text):
    """
    Complete preprocessing pipeline for clinical notes.
    
    Parameters:
    raw_text (str): Raw clinical note text
    
    Returns:
    dict: Dictionary containing all preprocessing results
    """
    # Step 1: Clean the text
    cleaned = clean_clinical_text(raw_text)
    
    # Step 2: Tokenize
    tokens = word_tokenize(cleaned)
    
    # Step 3: Remove stopwords and lemmatize
    processed_tokens = preprocess_tokens(tokens)
    
    # Step 4: Extract named entities from original text
    entities = extract_medical_entities(raw_text)
    
    # Return comprehensive results
    return {
        'cleaned_text': cleaned,
        'tokens': processed_tokens,
        'token_count': len(processed_tokens),
        'entities': entities,
        'entity_count': len(entities)
    }

# Test the function with a sample note
test_note = """Patient presents with acute dyspnea and chest pain.
History of CAD and recent MI. Started on aspirin and statin therapy."""

# Apply the full pipeline
results = full_preprocessing_pipeline(test_note)

# Display results
print("Preprocessing Pipeline Results:")
print("="*60)
print(f"Cleaned text: {results['cleaned_text']}")
print(f"\nToken count: {results['token_count']}")
print(f"Tokens: {results['tokens']}")
print(f"\nEntity count: {results['entity_count']}")
print("\nEntities extracted:")
for entity in results['entities']:
    print(f"  - {entity['text']} ({entity['label']})")
```

</details>

# Congratulations!

You have successfully completed this lab on text preprocessing for clinical notes. You learned how to clean and normalize raw clinical text, apply tokenization and lemmatization techniques, remove stopwords, and extract medical named entities using specialized NLP models. These foundational skills are essential for any clinical natural language processing application.

## Authors

Ramesh Sannareddy

Copyright © 2025 SkillUp. All rights reserved.
